### cover_nan.py에서 결측치를 해결한 train 데이터셋 가져오기

In [ ]:
import sys
import os
import pandas as pd

# 현재 작업 디렉토리 경로를 가져옵니다.
current_dir = os.getcwd()

# 노트북의 위치가 './다훈'이라면 상위 디렉토리로 이동하기 위해
# '../'를 사용하여 경로 조정. 'shared codes' 디렉토리 경로를 sys.path에 추가합니다.
shared_codes_dir = os.path.join(current_dir, '../shared codes')
sys.path.append(shared_codes_dir)

# 이제 cover_nan 모듈을 임포트할 수 있습니다.
from cover_nan import missing_value_removal_function

train = pd.read_csv("../shared codes/data/train.csv")
# missing_value_removal_function 사용
df = missing_value_removal_function(train)

## 전처리

### bool 컬럼 만들기

In [57]:
binary_columns = [col for col in df.columns if df[col].dropna().isin([0, 1]).all()]

print("0 또는 1만 포함된 컬럼:", binary_columns)

# 해당 컬럼들을 bool 타입으로 변환하기
for col in binary_columns:
    df[col] = df[col].astype(bool)

print(df.dtypes)

# df[binary_columns].head()

0 또는 1만 포함된 컬럼: ['배란 자극 여부', '단일 배아 이식 여부', '남성 주 불임 원인', '남성 부 불임 원인', '여성 주 불임 원인', '여성 부 불임 원인', '부부 주 불임 원인', '부부 부 불임 원인', '불명확 불임 원인', '불임 원인 - 난관 질환', '불임 원인 - 남성 요인', '불임 원인 - 배란 장애', '불임 원인 - 여성 요인', '불임 원인 - 자궁경부 문제', '불임 원인 - 자궁내막증', '불임 원인 - 정자 농도', '불임 원인 - 정자 면역학적 요인', '불임 원인 - 정자 운동성', '불임 원인 - 정자 형태', '배아 생성 주요 이유', '동결 배아 사용 여부', '신선 배아 사용 여부', '기증 배아 사용 여부', '대리모 여부', '난자 채취 경과일', '임신 성공 여부']
ID            object
시술 시기 코드      object
시술 당시 나이      object
시술 유형         object
특정 시술 유형      object
              ...   
대리모 여부          bool
난자 채취 경과일       bool
난자 혼합 경과일    float64
배아 이식 경과일    float64
임신 성공 여부        bool
Length: 62, dtype: object


### '정자 나이', '난자 나이' 컬럼 만들고 불필요한 컬럼 삭제

'시술 당시 나이', '정자 기증자 나이', '난자 기증자 나이', '난자 출처', '정자 출처' 컬럼을 삭제해도 된다.

In [58]:
import pandas as pd

def categorize_egg_age(row):
    # 난자 나이 통합
    egg_age = row['난자 기증자 나이'] if '난자 기증자 나이' in row else '알 수 없음'

    if egg_age in ['만21-25세', '만26-30세', '만31-35세', '만18-34세']:
        category = '건강한 난자'
    elif egg_age in ['만35-37세', '만38-39세']:
        category = '노화가 진행 중인 난자'
    elif egg_age in ['만40-42세', '만43-44세', '만45-50세']:
        category = '노화된 난자'
    else:
        category = '알 수 없음'
    
    return category

# 난자 나이 카테고리화 적용
df['난자 나이 카테고리'] = df.apply(categorize_egg_age, axis=1)

# '알 수 없음' 데이터 제거
df = df[df['난자 나이 카테고리'] != '알 수 없음']

# 불필요한 컬럼 삭제
df = df.drop(columns=['시술 당시 나이', '정자 기증자 나이', '난자 기증자 나이', '난자 출처', '정자 출처'])

df_young = df[df['난자 나이 카테고리'] == '건강한 난자'].drop(columns=['난자 나이 카테고리'])
df_middle = df[df['난자 나이 카테고리'] == '노화가 진행 중인 난자'].drop(columns=['난자 나이 카테고리'])
df_old = df[df['난자 나이 카테고리'] == '노화된 난자'].drop(columns=['난자 나이 카테고리'])

In [60]:
df_young.columns

# df_young의 컬럼 개수
print("건강한 난자 컬럼 개수:", len(df_young.columns))

건강한 난자 컬럼 개수: 57


In [55]:
print(df_young['임신 성공 여부'].value_counts())
print(df_middle['임신 성공 여부'].value_counts())
print(df_old['임신 성공 여부'].value_counts())


임신 성공 여부
False    76990
True     36737
Name: count, dtype: int64
임신 성공 여부
False    70435
True     23630
Name: count, dtype: int64
임신 성공 여부
False    42152
True      5784
Name: count, dtype: int64


## 시도하기

### 난자 연령별로 구분한 데이터프레임에서 bool 타입 컬럼들의 값 비율 알아보기

In [42]:
def calculate_bool_ratios(df, col):
    false_count = (df[col] == False).sum()  # False값의 수
    true_count = (df[col] == True).sum()    # True값의 수

    if true_count > 0:
        ratio = false_count / true_count
        return f"{false_count}:{true_count} -> {ratio:.2f}:1"
    else:
        return f"{false_count}:{true_count} -> Uncomparable (div by zero)"

# df의 bool 타입 컬럼
bool_columns = df.select_dtypes(include=bool).columns

# 각 연령대별 데이터프레임에 대해 비율 계산
df_age_groups = {
    "만18-34세": df_18_34,
    "만35-37세": df_35_37,
    "만38-39세": df_38_39,
    "만40-42세": df_40_42,
    "만43-44세": df_43_44,
    "만45-50세": df_45_50,
    "알 수 없음": df_unknown,
    "만20세 이하": df_20_under
}

# 각 컬럼에 대해 연령대별 비율 계산
column_ratios = {col: {} for col in bool_columns}

for col in bool_columns:
    for label, df_subset in df_age_groups.items():
        column_ratios[col][label] = calculate_bool_ratios(df_subset, col)


print("False : True 비율")

# 결과 출력
for col in bool_columns:
    print(f"Boolean ratios for column '{col}':")
    for label, ratio in column_ratios[col].items():
        print(f"  {label}: {ratio}")
    print()

False : True 비율
Boolean ratios for column '배란 자극 여부':
  만18-34세: 28101:85626 -> 0.33:1
  만35-37세: 10582:45617 -> 0.23:1
  만38-39세: 7570:30296 -> 0.25:1
  만40-42세: 7873:26551 -> 0.30:1
  만43-44세: 2711:7294 -> 0.37:1
  만45-50세: 1494:2013 -> 0.74:1
  알 수 없음: 6:323 -> 0.02:1
  만20세 이하: 294:0 -> Uncomparable (div by zero)

Boolean ratios for column '단일 배아 이식 여부':
  만18-34세: 77706:36021 -> 2.16:1
  만35-37세: 42665:13534 -> 3.15:1
  만38-39세: 32370:5496 -> 5.89:1
  만40-42세: 31839:2585 -> 12.32:1
  만43-44세: 9515:490 -> 19.42:1
  만45-50세: 3332:175 -> 19.04:1
  알 수 없음: 329:0 -> Uncomparable (div by zero)
  만20세 이하: 212:82 -> 2.59:1

Boolean ratios for column '남성 주 불임 원인':
  만18-34세: 111355:2372 -> 46.95:1
  만35-37세: 54441:1758 -> 30.97:1
  만38-39세: 36433:1433 -> 25.42:1
  만40-42세: 33117:1307 -> 25.34:1
  만43-44세: 9698:307 -> 31.59:1
  만45-50세: 3379:128 -> 26.40:1
  알 수 없음: 329:0 -> Uncomparable (div by zero)
  만20세 이하: 289:5 -> 57.80:1

Boolean ratios for column '남성 부 불임 원인':
  만18-34세: 112760:967

In [43]:
bool_columns

Index(['배란 자극 여부', '단일 배아 이식 여부', '남성 주 불임 원인', '남성 부 불임 원인', '여성 주 불임 원인',
       '여성 부 불임 원인', '부부 주 불임 원인', '부부 부 불임 원인', '불명확 불임 원인', '불임 원인 - 난관 질환',
       '불임 원인 - 남성 요인', '불임 원인 - 배란 장애', '불임 원인 - 여성 요인', '불임 원인 - 자궁경부 문제',
       '불임 원인 - 자궁내막증', '불임 원인 - 정자 농도', '불임 원인 - 정자 면역학적 요인',
       '불임 원인 - 정자 운동성', '불임 원인 - 정자 형태', '배아 생성 주요 이유', '동결 배아 사용 여부',
       '신선 배아 사용 여부', '기증 배아 사용 여부', '대리모 여부', '난자 채취 경과일', '임신 성공 여부'],
      dtype='object')

### IVF 환자와 DI 환자, 두 개의 데이터프레임으로 나누기

In [60]:
df_ivf = df[df['시술 유형'] == "IVF"]
df_di = df[df['시술 유형'] == "DI"]

df_ivf = df_ivf[interest_col]
df_di = df_di[interest_col]

### IVF 환자 데이터프레임에서 연관 컬럼 확인하기 w/ 도메인 지식

In [64]:
# 수치형 컬럼 확인하기
numeric_cols = df_ivf.select_dtypes(include=['int64', 'float64']).columns
numeric_cols

Index(['총 생성 배아 수', '미세주입된 난자 수', '미세주입에서 생성된 배아 수', '이식된 배아 수', '미세주입 배아 이식 수', '저장된 배아 수', '미세주입 후 저장된 배아 수', '해동된 배아 수', '해동 난자 수', '수집된 신선 난자 수', '저장된 신선 난자 수', '혼합된 난자 수', '파트너 정자와 혼합된 난자 수', '기증자 정자와 혼합된 난자 수', '난자 혼합 경과일', '배아 이식 경과일'], dtype='object')

In [13]:
# 범주형 컬럼 확인하기
categorical_cols = df_ivf.select_dtypes(include=['object']).columns
categorical_cols

Index(['시술 당시 나이', '시술 유형', '특정 시술 유형', '배란 유도 유형', '총 시술 횟수', '클리닉 내 총 시술 횟수',
       'IVF 시술 횟수', 'DI 시술 횟수', '총 임신 횟수', 'IVF 임신 횟수', 'DI 임신 횟수', '총 출산 횟수',
       'IVF 출산 횟수', 'DI 출산 횟수', '난자 출처', '정자 출처', '난자 기증자 나이', '정자 기증자 나이'],
      dtype='object')

In [65]:
# bool 컬럼 확인하기
bool_cols = df_ivf.select_dtypes(include=['bool']).columns
bool_cols

Index(['배란 자극 여부', '단일 배아 이식 여부', '남성 주 불임 원인', '남성 부 불임 원인', '여성 주 불임 원인', '여성 부 불임 원인', '부부 주 불임 원인', '부부 부 불임 원인', '불명확 불임 원인', '불임 원인 - 난관 질환', '불임 원인 - 남성 요인', '불임 원인 - 배란 장애', '불임 원인 - 여성 요인', '불임 원인 - 자궁경부 문제', '불임 원인 - 자궁내막증', '불임 원인 - 정자 농도', '불임 원인 - 정자 면역학적 요인', '불임 원인 - 정자 운동성', '불임 원인 - 정자 형태', '동결 배아 사용 여부', '신선 배아 사용 여부', '기증 배아 사용 여부', '대리모 여부', '난자 채취 경과일', '임신 성공 여부'], dtype='object')

### 주제: 불임 원인

In [62]:
# 주제: 불임 원인
causes_of_infertility_cols = ['남성 주 불임 원인',
 '남성 부 불임 원인',
 '여성 주 불임 원인',
 '여성 부 불임 원인',
 '부부 주 불임 원인',
 '부부 부 불임 원인',
 '불명확 불임 원인',
 '불임 원인 - 난관 질환',
 '불임 원인 - 남성 요인',
 '불임 원인 - 배란 장애',
 '불임 원인 - 여성 요인',
 '불임 원인 - 자궁경부 문제',
 '불임 원인 - 자궁내막증',
 '불임 원인 - 정자 농도',
 '불임 원인 - 정자 면역학적 요인',
 '불임 원인 - 정자 운동성',
 '불임 원인 - 정자 형태']

df_ivf_causes_of_infertility = df_ivf[causes_of_infertility_cols]


In [63]:
df_ivf_causes_of_infertility[df_ivf_causes_of_infertility['남성 주 불임 원인'] == 1][0:30]

,남성 주 불임 원인,남성 부 불임 원인,여성 주 불임 원인,여성 부 불임 원인,부부 주 불임 원인,부부 부 불임 원인,불명확 불임 원인,불임 원인 - 난관 질환,불임 원인 - 남성 요인,불임 원인 - 배란 장애,불임 원인 - 여성 요인,불임 원인 - 자궁경부 문제,불임 원인 - 자궁내막증,불임 원인 - 정자 농도,불임 원인 - 정자 면역학적 요인,불임 원인 - 정자 운동성,불임 원인 - 정자 형태
152,True,False,True,False,True,False,True,False,False,False,False,False,False,False,False,False,False
194,True,False,True,False,True,False,True,False,False,False,False,False,False,False,False,False,False
224,True,False,True,False,True,False,False,False,True,False,False,False,False,False,False,False,False
301,True,False,True,False,True,False,False,True,False,True,False,False,False,False,False,False,False
372,True,False,False,True,True,False,False,True,False,False,False,False,False,False,False,False,False
414,True,False,True,False,True,False,False,False,False,False,False,False,True,False,False,False,False
440,True,False,True,False,True,False,False,False,True,True,False,False,False,False,False,False,False
478,True,False,True,False,True,False,True,False,False,False,False,False,False,False,False,False,False
498,True,False,True,False,True,False,False,True,False,True,False,False,True,False,False,False,False
522,True,False,True,False,True,False,False,False,False,False,False,False,True,False,False,False,False


### 주제: 난자

In [67]:
temp = df_ivf[['시술 당시 나이', '난자 출처', '정자 출처', '난자 기증자 나이', '정자 기증자 나이', '대리모 여부', '난자 채취 경과일', '난자 혼합 경과일']]
temp[temp['난자 기증자 나이'] != temp['시술 당시 나이']]

,시술 당시 나이,난자 출처,정자 출처,난자 기증자 나이,정자 기증자 나이,대리모 여부,난자 채취 경과일,난자 혼합 경과일
6,만18-34세,기증 제공,배우자 제공,만21-25세,알 수 없음,False,False,0.0
22,만40-42세,기증 제공,배우자 제공,만21-25세,알 수 없음,False,False,0.0
40,만18-34세,기증 제공,기증 제공,만31-35세,만26-30세,False,False,0.0
79,만45-50세,기증 제공,배우자 제공,만26-30세,알 수 없음,False,False,0.0
100,만18-34세,기증 제공,배우자 제공,만26-30세,알 수 없음,False,False,0.0
116,만45-50세,기증 제공,기증 제공,만31-35세,만21-25세,False,False,0.0
129,만18-34세,기증 제공,배우자 제공,만31-35세,알 수 없음,False,False,0.0
204,만43-44세,기증 제공,배우자 제공,만26-30세,알 수 없음,False,False,0.0
206,만18-34세,기증 제공,기증 제공,만31-35세,만41-45세,True,False,0.0
230,만45-50세,기증 제공,배우자 제공,만26-30세,알 수 없음,False,False,0.0


In [26]:
df_ivf[['배란 자극 여부', '배란 유도 유형', '단일 배아 이식 여부']][0:30]
df_ivf[['배란 자극 여부', '배란 유도 유형', '단일 배아 이식 여부']][0:30]


,배란 자극 여부,배란 유도 유형,단일 배아 이식 여부
0,1,기록되지 않은 시행,0.0
1,0,알 수 없음,0.0
2,1,기록되지 않은 시행,0.0
3,1,기록되지 않은 시행,0.0
4,1,기록되지 않은 시행,0.0
5,1,기록되지 않은 시행,0.0
6,0,알 수 없음,1.0
7,1,기록되지 않은 시행,0.0
8,1,기록되지 않은 시행,0.0
9,1,기록되지 않은 시행,0.0


In [49]:
df_embryo = df_ivf[['단일 배아 이식 여부', '이식된 배아 수', '임신 성공 여부']]
df_embryo[(df_embryo['이식된 배아 수'] == 0.0) & (df_embryo['단일 배아 이식 여부'] != 0)][0:50]
df_embryo.info()

<class 'pandas.core.frame.DataFrame'>
Index: 250060 entries, 0 to 256350
Data columns (total 3 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   단일 배아 이식 여부  250060 non-null  float64
 1   이식된 배아 수     250060 non-null  float64
 2   임신 성공 여부     250060 non-null  int64  
dtypes: float64(2), int64(1)
memory usage: 7.6 MB


In [45]:
df_embryo[df_embryo['단일 배아 이식 여부'] != 0]['이식된 배아 수'].value_counts()


이식된 배아 수
1.0    58360
2.0       21
3.0        2
Name: count, dtype: int64

In [46]:
df_embryo[df_embryo['단일 배아 이식 여부'] != 0].value_counts()


단일 배아 이식 여부  이식된 배아 수  임신 성공 여부
1.0          1.0       0           36926
                       1           21434
             2.0       0              15
                       1               6
             3.0       0               2
Name: count, dtype: int64

In [47]:
df_ivf['임신 성공 여부'].value_counts()

임신 성공 여부
0    184643
1     65417
Name: count, dtype: int64